Análisis de Caso: Reducción Dimensional

Lección: Reducción dimensional — VisionData

Contexto del problema

VisionData trabaja con encuestas masivas de clientes que contienen más de 100 variables por registro. La alta dimensionalidad genera tiempos de entrenamiento elevados, bajo rendimiento predictivo y visualizaciones poco interpretables para las áreas de marketing y ventas. El objetivo de este análisis es aplicar PCA y t-SNE sobre el dataset de encuestas de Seaborn, comparar ambas técnicas y recomendar la más adecuada para presentar insights al equipo de negocio.

1. Carga de librerías y dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)  # semilla para reproducibilidad

# cargamos el dataset 'penguins' de seaborn: 4 variables numéricas continuas
# y una variable categórica (especie) que usaremos como etiqueta de color en las visualizaciones
df_raw = sns.load_dataset('penguins')

print("Dataset cargado — primeras filas:")
print(df_raw.head(8).to_string(index=False))
print(f"\n Shape del dataset original: {df_raw.shape}")
print("\n Tipos de variables:")
print(df_raw.dtypes)
print("\n Distribución por especie:")
print(df_raw['species'].value_counts())

Optamos por el dataset `penguins` de Seaborn porque posee múltiples variables morfológicas numéricas continuas (longitud/profundidad del pico, longitud de aleta, masa corporal) más una variable categórica de especie que nos permite validar visualmente si las técnicas de reducción dimensional logran separar los grupos naturales. Esto replica la lógica del caso VisionData: variables cuantitativas de clientes + segmentos conocidos que debemos recuperar.

2. Exploración y limpieza de datos

In [ ]:
#2.1 Resumen estadístico y detección de nulos

print("Resumen estadístico — variables numéricas:")
print(df_raw.describe().round(2).to_string())
print("\n Valores nulos por columna:")
print(df_raw.isnull().sum())

In [ ]:
#2.2 Limpieza: eliminamos filas con nulos y nos quedamos solo con las variables numéricas

features = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']

df_clean = df_raw.dropna(subset=features + ['species']).reset_index(drop=True)
labels   = df_clean['species']   # guardamos las etiquetas para colorear los gráficos
X        = df_clean[features]    # matriz de características

print(f"Filas antes de limpiar : {len(df_raw)}")
print(f"Filas después de limpiar: {len(df_clean)}")
print(f"Filas eliminadas (nulos): {len(df_raw) - len(df_clean)}")
print(f"\n Variables utilizadas: {features}")
print(f" Dimensionalidad de X  : {X.shape}")

In [ ]:
#2.3 Escalado de variables
# PCA y t-SNE son sensibles a la escala: sin estandarizar, variables con rangos
# grandes (como body_mass_g) dominarían la varianza y distorsionarían los resultados.

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Variables escaladas (media ≈ 0, std ≈ 1) — estadísticas post-escalado:")
df_scaled_check = pd.DataFrame(X_scaled, columns=features)
print(df_scaled_check.describe().round(3).to_string())

El dataset quedó con 342 registros válidos luego de eliminar 2 filas con nulos. Las cuatro variables numéricas fueron estandarizadas con StandardScaler (media=0, std=1) para garantizar que ninguna domine la varianza por cuestión de escala, lo que es especialmente importante para PCA.

3. Aplicación de PCA

In [ ]:
#3.1 PCA completo: primero analizamos cuánta varianza explica cada componente

pca_full = PCA(random_state=42)
pca_full.fit(X_scaled)

varianza_exp       = pca_full.explained_variance_ratio_
varianza_acumulada = np.cumsum(varianza_exp)

print("Varianza explicada por cada componente principal:")
for i, (v, va) in enumerate(zip(varianza_exp, varianza_acumulada), 1):
    print(f"  PC{i}: {v:.4f}  ({v*100:.1f}%)  — acumulado: {va*100:.1f}%")
print(f"\n Con 2 componentes se retiene el {varianza_acumulada[1]*100:.1f}% de la varianza total.")

In [ ]:
#3.2 Gráfico de varianza explicada (Scree plot)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
componentes = [f'PC{i}' for i in range(1, len(varianza_exp)+1)]

# varianza individual por componente
axes[0].bar(componentes, varianza_exp * 100, color='steelblue', edgecolor='white')
axes[0].set_title('Varianza explicada por componente', fontsize=12)
axes[0].set_ylabel('Varianza explicada (%)')
axes[0].set_xlabel('Componente Principal')

# varianza acumulada
axes[1].plot(componentes, varianza_acumulada * 100, marker='o', color='steelblue', linewidth=2)
axes[1].axhline(y=90, color='tomato', linestyle='--', linewidth=1, label='Umbral 90%')
axes[1].set_title('Varianza acumulada', fontsize=12)
axes[1].set_ylabel('Varianza acumulada (%)')
axes[1].set_xlabel('Componente Principal')
axes[1].legend()
axes[1].set_ylim(0, 105)

plt.tight_layout()
plt.savefig('scree_plot.png', dpi=120, bbox_inches='tight')
plt.show()
print("Gráfico guardado como scree_plot.png")

In [ ]:
#3.3 Reducción a 2 componentes

pca_2d = PCA(n_components=2, random_state=42)
X_pca  = pca_2d.fit_transform(X_scaled)

df_pca = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
df_pca['species'] = labels.values

print("Proyección PCA — primeras filas:")
print(df_pca.head(6).to_string(index=False))
print(f"\n Varianza explicada por PC1: {pca_2d.explained_variance_ratio_[0]*100:.1f}%")
print(f" Varianza explicada por PC2: {pca_2d.explained_variance_ratio_[1]*100:.1f}%")
print(f" Total retenido con 2 PCs  : {sum(pca_2d.explained_variance_ratio_)*100:.1f}%")

In [ ]:
#3.4 Visualización 2D — PCA

colores = {'Adelie': '#4C72B0', 'Chinstrap': '#DD8452', 'Gentoo': '#55A868'}

fig, ax = plt.subplots(figsize=(9, 6))

for especie, grupo in df_pca.groupby('species'):
    ax.scatter(grupo['PC1'], grupo['PC2'],
               label=especie, color=colores[especie],
               alpha=0.75, edgecolors='white', linewidths=0.5, s=60)

ax.set_title('PCA — Proyección 2D de Pingüinos por Especie', fontsize=14, pad=12)
ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% varianza explicada)', fontsize=11)
ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% varianza explicada)', fontsize=11)
ax.legend(title='Especie', fontsize=10)
ax.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('visualizacion_pca.png', dpi=150, bbox_inches='tight')
plt.show()
print("Gráfico guardado como visualizacion_pca.png")

In [ ]:
#3.5 Loadings de PCA: ¿qué variables contribuyen más a cada componente?

df_loadings = pd.DataFrame(
    pca_2d.components_.T,
    index=features,
    columns=['PC1', 'PC2']
).round(4)

print("Loadings (contribución de cada variable original a los componentes):")
print(df_loadings.to_string())

Con solo 2 componentes principales, PCA retiene aproximadamente el 88% de la varianza total del dataset. PC1 representa principalmente el tamaño corporal (aleta y masa con cargas altas y del mismo signo), mientras que PC2 captura la morfología del pico (longitud vs. profundidad con cargas opuestas). La visualización 2D muestra una separación clara entre las tres especies, especialmente Gentoo, lo que sugiere que las variables originales contienen estructura lineal explotable.

4. Aplicación de t-SNE

In [ ]:
#4.1 t-SNE con hiperparámetros estándar
# perplexity: controla el balance entre estructura local y global (recomendado: 5–50)
# n_iter: número de iteraciones de optimización
# learning_rate: tasa de aprendizaje para la optimización de la función de costo KL-divergencia
# init='pca': inicialización con PCA para mayor estabilidad y reproducibilidad

tsne = TSNE(n_components=2,
            perplexity=30,
            n_iter=1000,
            learning_rate='auto',
            init='pca',
            random_state=42)

X_tsne = tsne.fit_transform(X_scaled)

df_tsne = pd.DataFrame(X_tsne, columns=['Dim1', 'Dim2'])
df_tsne['species'] = labels.values

print("Proyección t-SNE — primeras filas:")
print(df_tsne.head(6).to_string(index=False))
print(f"\n KL-divergencia final (menor = mejor): {tsne.kl_divergence_:.4f}")

A diferencia de PCA, las coordenadas de t-SNE no tienen interpretación geométrica directa: no representan varianza explicada ni permiten comparar distancias entre clusters de forma absoluta. Su valor está exclusivamente en la visualización de la estructura local.

In [ ]:
#4.2 Visualización 2D — t-SNE

fig, ax = plt.subplots(figsize=(9, 6))

for especie, grupo in df_tsne.groupby('species'):
    ax.scatter(grupo['Dim1'], grupo['Dim2'],
               label=especie, color=colores[especie],
               alpha=0.75, edgecolors='white', linewidths=0.5, s=60)

ax.set_title('t-SNE — Proyección 2D de Pingüinos por Especie', fontsize=14, pad=12)
ax.set_xlabel('Dimensión 1 (t-SNE)', fontsize=11)
ax.set_ylabel('Dimensión 2 (t-SNE)', fontsize=11)
ax.legend(title='Especie', fontsize=10)
ax.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('visualizacion_tsne.png', dpi=150, bbox_inches='tight')
plt.show()
print("Gráfico guardado como visualizacion_tsne.png")

t-SNE logra clusters más compactos y visualmente separados que PCA, especialmente para las especies Adelie y Chinstrap que en PCA presentaban cierto solapamiento. Esto se debe a que t-SNE optimiza directamente la estructura local de los datos preservando vecindades, a costo de perder la interpretabilidad de los ejes y la estructura global.

5. Comparación entre PCA y t-SNE

In [ ]:
#5.1 Visualización lado a lado para comparación directa

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- PCA ---
for especie, grupo in df_pca.groupby('species'):
    axes[0].scatter(grupo['PC1'], grupo['PC2'],
                    label=especie, color=colores[especie],
                    alpha=0.75, edgecolors='white', linewidths=0.5, s=60)

axes[0].set_title('PCA — 2 Componentes Principales', fontsize=13)
axes[0].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% var. expl.)')
axes[0].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% var. expl.)')
axes[0].legend(title='Especie', fontsize=9)
axes[0].grid(True, linestyle='--', alpha=0.4)
axes[0].text(0.02, 0.97,
             f'Varianza retenida: {sum(pca_2d.explained_variance_ratio_)*100:.1f}%',
             transform=axes[0].transAxes, fontsize=9,
             verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7))

# --- t-SNE ---
for especie, grupo in df_tsne.groupby('species'):
    axes[1].scatter(grupo['Dim1'], grupo['Dim2'],
                    label=especie, color=colores[especie],
                    alpha=0.75, edgecolors='white', linewidths=0.5, s=60)

axes[1].set_title('t-SNE — perplexity=30, n_iter=1000', fontsize=13)
axes[1].set_xlabel('Dimensión 1 (t-SNE)')
axes[1].set_ylabel('Dimensión 2 (t-SNE)')
axes[1].legend(title='Especie', fontsize=9)
axes[1].grid(True, linestyle='--', alpha=0.4)
axes[1].text(0.02, 0.97,
             f'KL-divergencia: {tsne.kl_divergence_:.4f}',
             transform=axes[1].transAxes, fontsize=9,
             verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7))

plt.suptitle('Comparación PCA vs t-SNE — Dataset Penguins', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('comparacion_pca_tsne.png', dpi=150, bbox_inches='tight')
plt.show()
print("Gráfico guardado como comparacion_pca_tsne.png")

In [ ]:
#5.2 Tabla comparativa de características clave

comparacion = {
    'Criterio': [
        'Tipo de relaciones que captura',
        'Ejes interpretables',
        'Varianza explicada cuantificable',
        'Separación de clusters',
        'Determinismo (mismo resultado siempre)',
        'Escalabilidad a grandes volúmenes',
        'Sensibilidad a hiperparámetros',
        'Uso como preprocesamiento para ML',
        'Adecuado para presentaciones ejecutivas'
    ],
    'PCA': [
        'Lineales', 'Sí (loadings)', 'Sí', 'Moderada', 'Sí',
        'Alta (O(n·d²))', 'Baja', 'Sí (reduce dimensiones de entrada)', 'Sí'
    ],
    't-SNE': [
        'No lineales', 'No (ejes sin significado)', 'No', 'Alta', 'No (estocástico)',
        'Baja (O(n²))', 'Alta (perplexity, lr)', 'No (solo visualización)', 'Con precauciones'
    ]
}

df_comp = pd.DataFrame(comparacion)
print(df_comp.to_string(index=False))

La comparación visual revela que t-SNE produce clusters más compactos y separados, pero PCA ofrece algo que t-SNE no puede: ejes con significado (PC1 = tamaño corporal, PC2 = morfología del pico) y una métrica de calidad objetiva (varianza retenida = 88%). Para una presentación ejecutiva en marketing, PCA es más transparente y confiable; t-SNE es útil para exploración analítica interna.

6. Justificación de técnica recomendada y reflexión crítica

6.1 Técnica recomendada para VisionData: PCA

Razones:

1. Interpretabilidad: los ejes de PCA tienen significado concreto. En el contexto de encuestas de clientes, PC1 podría representar 'nivel de engagement digital' y PC2 'sensibilidad al precio', por ejemplo.
2. Reproducibilidad: el resultado es determinístico. Ejecutar el mismo código en distintos momentos produce exactamente la misma visualización.
3. Cuantificación: se puede reportar al área de marketing qué porcentaje de la información original se retiene (en este caso, ~88%).
4. Escalabilidad: con 100+ variables y miles de clientes, PCA escala mejor que t-SNE (cuya complejidad es cuadrática).
5. Preprocesamiento: las componentes pueden usarse como input directo para modelos predictivos posteriores.

Cuándo usar t-SNE en cambio:
- Exploración interna del equipo de datos para detectar clusters no lineales antes de aplicar otros algoritmos.
- Validación visual de resultados de clustering (K-Means, DBSCAN).
- Datasets medianos (< 10.000 filas) donde el costo computacional es tolerable.

6.2 Reflexión crítica: limitaciones y escalabilidad

PCA:
- Solo captura relaciones lineales entre variables. Si los segmentos de clientes tienen fronteras no lineales, PCA puede mezclarlos en la proyección 2D.
- Las componentes principales son combinaciones de TODAS las variables originales, lo que dificulta su interpretación cuando se trabaja con 100+ variables.
- La varianza no es necesariamente relevancia: una variable con alta varianza no es necesariamente la más importante para el negocio.

t-SNE:
- Las distancias entre clusters NO son comparables: dos clusters muy separados visualmente no significan que sean 'más distintos' que dos clusters cercanos.
- Resultados estocásticos: con distinto random_state o perplexity la visualización puede cambiar significativamente.
- Complejidad O(n²): con 100.000+ clientes, t-SNE convencional es inviable en tiempo razonable.

¿Qué haría con un volumen de datos mucho mayor?
- Aplicaría PCA como primer paso para reducir de 100+ a ~20 dimensiones.
- Luego usaría UMAP en lugar de t-SNE: más rápido, determinístico con semilla fija, preserva mejor la estructura global.
- Para escala industrial (millones de registros), consideraría Incremental PCA o técnicas de compresión con redes neuronales (Autoencoders).

Conclusiones

Se evaluaron dos técnicas de reducción dimensional sobre un dataset de características morfológicas de pingüinos (proxy del caso de encuestas de VisionData). Ambas técnicas lograron separar las tres especies en el espacio 2D, pero con diferencias importantes.

PCA retuvo el 88% de la varianza original con solo 2 componentes y produjo ejes interpretables (PC1 = tamaño corporal, PC2 = morfología del pico). t-SNE generó clusters visualmente más compactos, especialmente para las especies con mayor solapamiento, pero sin ejes interpretables y con comportamiento estocástico.

Recomendación para VisionData: para presentar insights al equipo de marketing se recomienda PCA por su interpretabilidad, reproducibilidad y escalabilidad. t-SNE es valioso como herramienta exploratoria interna del equipo de datos antes de decidir estrategias de segmentación. Ante volúmenes masivos de clientes, la combinación PCA + UMAP representa la alternativa más robusta.